# SmartHeart Colab Training

## 1. Clone the repository

Clone the latest `main` branch into the Colab workspace. If it is already present, update it with a fast-forward pull, then make it the active working directory.

In [1]:
!if [ -d /content/smart-heart/.git ]; then git -C /content/smart-heart pull --ff-only origin main; else git clone --branch main --single-branch https://github.com/Bojan-Ivanovski/smart-heart.git /content/smart-heart; fi
%cd /content/smart-heart


Cloning into '/content/smart-heart'...
remote: Enumerating objects: 2001, done.
remote: Counting objects: 100% (1204/1204), done.
remote: Compressing objects: 100% (350/350), done.
remote: Total 2001 (delta 866), reused 1183 (delta 853), pack-reused 797 (from 2)
Receiving objects: 100% (2001/2001), 119.98 MiB | 31.90 MiB/s, done.
Resolving deltas: 100% (1143/1143), done.
/content/smart-heart


## 2. Configure the run

Select an **A100 GPU** from **Runtime > Change runtime type** for the quality-focused configuration below. The 1B Gemma model, SP architecture, LoRA, and cognitive-function source provide a practical full-curriculum run. `WINDOW_SIZE` controls the signal samples presented per model window; 2048 is the A100-oriented default. Use `cuda` for an A100 or L4, `xla` for TPU, or `cpu` only for functional checks.

In [11]:
import os

DEVICE = "cuda"  # @param ["cuda", "xla", "cpu"]
DATASET = "adhd_cognitive_function"  # @param ["adhd_cognitive_function", "adhd_children", "adhd_gameplay", "all"]
MODEL_ID = "google/gemma-3-1b-pt"  # @param {type:"string"}
WINDOW_SIZE = 4096  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
FRESH_START = False  # @param {type:"boolean"}

os.environ["SMARTHEART_DEVICE"] = DEVICE
os.environ["SMARTHEART_DATASET"] = DATASET
os.environ["SMARTHEART_MODEL_ID"] = MODEL_ID
os.environ["SMARTHEART_WINDOW_SIZE"] = str(WINDOW_SIZE)
os.environ["SMARTHEART_SEED"] = str(SEED)
os.environ["SMARTHEART_FRESH_START"] = "1" if FRESH_START else "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print(f"device={DEVICE}, dataset={DATASET}, model={MODEL_ID}, window_size={WINDOW_SIZE}")


device=cuda, dataset=adhd_cognitive_function, model=google/gemma-3-1b-pt, window_size=4096


## 3. Install dependencies

Upgrade `pip`, remove Colab's unused `diffusers` and `gradio` packages that conflict with OpenTSLM's Hugging Face stack, and install the repository's pinned environment. CUDA and CPU runs use the base requirements; XLA runs additionally install the TPU runtime and its official wheel source.

In [7]:
%%bash
set -euo pipefail
python -m pip install --upgrade pip
python -m pip uninstall --yes diffusers gradio
if [[ "${SMARTHEART_DEVICE}" == "xla" ]]; then
    python -m pip install -r requirements-tpu.txt
else
    python -m pip install -r requirements.txt
fi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 51.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Found existing installation: diffusers 0.40.0
Uninstalling diffusers-0.40.0:
  Successfully uninstalled diffusers-0.40.0
Found existing installation: gradio 6.26.0
Uninstalling gradio-6.26.0:
  Successfully uninstalled gradio-6.26.0
  Cloning https://github.com/StanfordBDHG/OpenTSLM.git (to revision refs/pull/47/head) to /tmp/pip-install-pn9crqe4/opentslm_bdc62bb0a1a9421db77aeac3f364273e
  Did not find branch or tag 'refs/pull/47/head', assuming revision or ref.
  Resolved https://github.com/StanfordBDHG/OpenTSLM.git to commit 3356bbd4ecd2a88f713784d3e14f71a2b5b5b2f2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with stat

  Running command git clone --filter=blob:none --quiet https://github.com/StanfordBDHG/OpenTSLM.git /tmp/pip-install-pn9crqe4/opentslm_bdc62bb0a1a9421db77aeac3f364273e
  Running command git fetch -q https://github.com/StanfordBDHG/OpenTSLM.git refs/pull/47/head
  Running command git checkout -q 3356bbd4ecd2a88f713784d3e14f71a2b5b5b2f2


## 4. Validate the dataset

Verify the canonical dataset cloned with the repository and report its patient and signal-archive counts. No extraction or build step is required.

In [8]:
%%bash
set -euo pipefail
test -d dataset/patients
echo "Patients: $(find dataset/patients -mindepth 1 -maxdepth 1 -type d | wc -l)"
echo "Signal archives: $(find dataset/patients -type f -name 'signals.npz' | wc -l)"


Patients: 210
Signal archives: 210


## 5. Validate the environment and preview data

Report the installed PyTorch version and CUDA availability, then preview the selected dataset to confirm discovery and loading before model training begins.

In [4]:
!python -c "import torch; print('torch:', torch.__version__); print('cuda_available:', torch.cuda.is_available())"
!python -m src.main preview --source {DATASET} --window-size {WINDOW_SIZE} --stage stage1_mcq


torch: 2.11.0+cu128
cuda_available: True
stage: stage1_mcq
stage_samples: 10252
signal_windows: 5126
patient: cognitive-function__fadhd__001
recordings: 01-eyes_open_baseline_cz_f4
segments: 01-eyes_open_baseline_cz_f4__segment-001
split: train
source: adhd_cognitive_function

PRE-PROMPT
Analyze the EEG time series and answer the multiple-choice question using the measured signal pattern.

TIME SERIES
[1] shape=(2048,) Recording 01-eyes_open_baseline_cz_f4, channel Cz, contains raw_eeg from the eyes_open_baseline_cz_f4 condition. Its original mean is -0.1836 and standard deviation is 23.9189.
[2] shape=(2048,) Recording 01-eyes_open_baseline_cz_f4, channel F4, contains raw_eeg from the eyes_open_baseline_cz_f4 condition. Its original mean is 0.5518 and standard deviation is 15.0422.

POST-PROMPT
Question: Among the listed options, which frequency band has the greatest relative power in this window?
A. Delta
B. Theta
C. Beta
D. Gamma
Answer with only A, B, C, or D.

REFERENCE ANALYSIS (

## 6. Authenticate with Hugging Face

Gemma access may require accepting its license on Hugging Face. Add a Colab secret named `HF_TOKEN` using the key icon in the left sidebar and grant this notebook access. The next cell stores it in Hugging Face's local cache without printing it.

In [5]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("HF_TOKEN is not configured. Public models can still be downloaded.")


Hugging Face authentication configured.


## 7. Train the model

Train each curriculum stage separately with quality-oriented epochs, batch sizes, and learning rates. Earlier stages learn signal interpretation from abundant window examples; later stages use smaller learning rates and additional passes over their smaller recording- and patient-level datasets. Every stage uses the configured `WINDOW_SIZE`. Each command initializes from the preceding stage checkpoint. Set `FRESH_START` when changing the window size for an existing model run. `FRESH_START` applies only to Stage 1, and LoRA remains enabled. The cell stops immediately if any stage fails.

In [14]:
import os
import subprocess
import sys

if DEVICE == "xla":
    os.environ["PJRT_DEVICE"] = "TPU"
os.environ["PYTHONUNBUFFERED"] = "1"

common_args = [
    sys.executable, "-u", "-m", "src.main", "train",
    "--model-id", MODEL_ID,
    "--architecture", "sp",
    "--source", DATASET,
    "--device", DEVICE,
    "--window-size", str(WINDOW_SIZE),
    "--seed", str(SEED),
]
stages = [
    # ("Stage 1: MCQ warmup", "stage1_mcq", 4, 2, "5e-5"),
    # ("Stage 2: EEG captioning", "stage2_eeg_captioning", 2, 3, "5e-5"),
    ("Stage 3: attention task reasoning", "stage3_attention_task_cot", 1, 8, "2e-5"),
    ("Stage 4: resting-state reasoning", "stage4_resting_state_cot", 1, 4, "2e-5"),
    ("Stage 5: diagnostic reasoning", "stage5_diagnostic_cot", 1, 10, "1e-5"),
]

for stage_index, (label, stage, batch_size, epochs, learning_rate) in enumerate(stages):
    command = common_args + [
        "--stage", stage,
        "--batch-size", str(batch_size),
        "--epochs", str(epochs),
        "--learning-rate", learning_rate,
    ]
    if stage_index == 0 and FRESH_START:
        command.append("--fresh-start")
    print(f"[notebook] Starting {label}", flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


[notebook] Starting Stage 1: MCQ warmup
2026-09-05 22:05:17.533582: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-05 22:05:17.607050: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
✅ LoRA enabled:
   LoRA parame

CalledProcessError: Command '['/usr/bin/python3', '-u', '-m', 'src.main', 'train', '--model-id', 'google/gemma-3-1b-pt', '--architecture', 'sp', '--source', 'adhd_cognitive_function', '--device', 'cuda', '--window-size', '4096', '--seed', '42', '--stage', 'stage3_attention_task_cot', '--batch-size', '1', '--epochs', '8', '--learning-rate', '2e-5']' returned non-zero exit status 1.

## 8. Inspect checkpoints

List every checkpoint artifact produced by training together with its allocated file size.

In [ ]:
!find checkpoints -type f -printf '%p (%k KB)\n'


## 9. Validate and test the checkpoint

Load the curriculum checkpoints produced above and evaluate every stage on the held-out validation and test patient partitions using the same configured window size. Each stage reports metrics appropriate to its target, and detailed predictions are written under `results/`.

In [ ]:
%%bash
set -euo pipefail
python -m src.main evaluate --source "${SMARTHEART_DATASET}" --window-size "${SMARTHEART_WINDOW_SIZE}" --device "${SMARTHEART_DEVICE}" --model-id "${SMARTHEART_MODEL_ID}" --split validation --batch-size 1
python -m src.main evaluate --source "${SMARTHEART_DATASET}" --window-size "${SMARTHEART_WINDOW_SIZE}" --device "${SMARTHEART_DEVICE}" --model-id "${SMARTHEART_MODEL_ID}" --split test --batch-size 1
